# boop: build the training data (free, Kaggle)

**Settings (right panel):** Accelerator **GPU T4 x1** · Internet **On** · Add-ons → Secrets → **`HF_TOKEN`** (write token).

1. Run with `LIMIT = 50` first (~15 min with installs): check the spot-check cell looks sensible.
2. Set `LIMIT = 0`, then **Save Version → Save & Run All (Commit)** so it runs in the background (~1–1.5 h, mostly the teacher).
   The teacher step is resumable: if the session dies, just run again.

Pipeline: tldr-pages → filled commands → **Qwen2.5-Coder-7B** writes casual requests (kept only if they mention every
file/value) → + synthetic tasks, each **executed and verified** → train/valid JSONL → private HF dataset.

In [ ]:
# ---- config ----------------------------------------------------------------------------
GIT_REPO      = "https://github.com/thebunnyweb/boop"
HF_DATA_REPO  = "bunnyweb/boop-data"        # private dataset repo for the result
CAPS          = "common=3000,osx=1200,linux=1500,windows=900"   # tldr examples per folder (× OS for common)
PER_TEMPLATE  = 60                          # synthetic examples per template per OS (42 templates × 3 OS)
LIMIT         = 50                          # teacher smoke test; 0 = everything

In [ ]:
import os, subprocess, json, random
WORK = "/kaggle/working/boop"
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", "-q", GIT_REPO, WORK], check=True)
os.chdir(WORK)
!git pull -q && git log --oneline -1
if not os.path.exists("data/tldr-src"):
    !git clone -q --depth 1 https://github.com/tldr-pages/tldr data/tldr-src
!pip install -q unsloth
!pip install -q -e . --no-deps             # boop itself; llama-cpp isn't needed here
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN"); print("HF_TOKEN loaded")

## 1. tldr-pages → concrete commands

In [ ]:
!python data/tldr.py data/tldr-src --out data/out/tldr.jsonl --max-per-platform {CAPS}

## 2. Teacher writes the requests (the slow part)

In [ ]:
lim = f"--limit {LIMIT}" if LIMIT else ""
!python data/teach.py --in data/out/tldr.jsonl --out data/out/tldr_requests.jsonl --batch 16 {lim}

In [ ]:
# spot-check: do these read like things a person would type, and does each command answer them?
rows = [json.loads(l) for l in open("data/out/tldr_requests.jsonl")]
print(f"{len(rows)} examples, {sum(bool(r['requests']) for r in rows)} with ≥1 valid request, "
      f"{sum(len(r['requests']) for r in rows)} requests\n")
for r in random.Random(1).sample(rows, min(15, len(rows))):
    print(f"[{r['os']}] {r['command']}")
    for q in r["requests"] or ["(none kept)  raw: " + r["raw"][:150]]:
        print("   •", q)

## 3. Synthetic tasks, each executed and verified

In [ ]:
!python data/synth.py --out data/out/synth.jsonl --per-template {PER_TEMPLATE}

## 4. Build train/valid and upload

In [ ]:
!python data/build.py --tldr data/out/tldr_requests.jsonl --synth data/out/synth.jsonl --out data/out/master
if LIMIT:
    print("smoke run (LIMIT set): not uploading")
else:
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ["HF_TOKEN"])
    api.create_repo(HF_DATA_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path="data/out/master", repo_id=HF_DATA_REPO, repo_type="dataset", path_in_repo="master")
    api.upload_folder(folder_path="data/out", repo_id=HF_DATA_REPO, repo_type="dataset", path_in_repo="raw",
                      allow_patterns=["*.jsonl"])
    print(f"uploaded → https://huggingface.co/datasets/{HF_DATA_REPO}")